# Create vector store for our system

In [5]:
# Load dataset 
from datasets import load_dataset

ds_path = "./wikipedia-source"
ds = load_dataset(ds_path)
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 208279
    })
})

In [7]:
# 1. Remove empty
ds = ds.filter(lambda s: s["text"].strip() != "")

# 2. Remove low content
ds = ds.filter(lambda s: len(s["text"].split()) > 15)

# 3. Remove duplicates
seen = set()
def is_unique(sample):
    text = sample["text"].strip()
    if text in seen:
        return False
    seen.add(text)
    return True

ds = ds.filter(is_unique)

Filter: 100%|██████████| 206188/206188 [00:03<00:00, 58075.66 examples/s]


In [1]:
from vector_store_module import Retriever

retriever = Retriever()


/home/parsa/.conda/envs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded index successfully! ./faiss_index/index


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("../models/embeddinggemma-300m")

def chunk_text_by_tokens(text: str, max_tokens: int = 2047):
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        sub_tokens = tokens[i:i + max_tokens]
        chunk = tokenizer.decode(sub_tokens, skip_special_tokens=True)
        chunks.append(chunk)
    return chunks

In [11]:
from tqdm import tqdm

buffer = []
batch_size = 140

for sample in tqdm(ds['train']):
    text = sample['text']

    # chunk the text
    chunks = chunk_text_by_tokens(text, max_tokens=2048)  # or chunk_text_by_words

    # add chunks into your batch buffer
    for ch in chunks:
        buffer.append(ch)

        # flush when batch is full
        if len(buffer) == batch_size:
            retriever.fast_add_documents(buffer, batch_size=batch_size)
            buffer = []

# add remaining documents
if buffer:
    retriever.fast_add_documents(buffer, batch_size=batch_size)

100%|██████████| 205668/205668 [3:01:50<00:00, 18.85it/s]  


In [12]:
retriever.save()

In [2]:
retriever.clean_empty_documents()

[INFO] Cleaning empty documents...


[INFO] Original documents: 418383
[INFO] Cleaned documents: 417571


Batches: 100%|██████████| 6525/6525 [2:11:55<00:00,  1.21s/it]  


[INFO] Cleanup complete.


In [3]:
retriever.save()

In [ ]:
retriever.search('steam engine', 2)

[' the automobile on the farm is the way that it has\nbroadened the farmer\'s life. We simply took for granted that unless the\nerrand were urgent we would not go to town, and I think we rarely made\nmore than a trip a week. In bad weather we did not go even that often.\n\nBeing a full-fledged machinist and with a very fair workshop on the farm\nit was not difficult for me to build a steam wagon or tractor. In the\nbuilding of it came the idea that perhaps it might be made for road use.\nI felt perfectly certain that horses, considering all the bother of\nattending them and the expense of feeding, did not earn their keep. The\nobvious thing to do was to design and build a steam engine that would be\nlight enough to run an ordinary wagon or to pull a plough. I thought it\nmore important first to develop the tractor. To lift farm drudgery off\nflesh and blood and lay it on steel and motors has been my most constant\nambition. It was circumstances that took me first into the actual\nmanuf

: 